> **中文阅读版**：本文件是 `03_state_and_memory.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# Agent 状态、记忆与质量门禁

**Docker 镜像**：`ml4t`

[`01_react_reasoning`](01_react_reasoning.ipynb) 中的 agent 把它知道的一切都保存在消息历史中。这足以产生一个答案，却不足以捍卫一个答案：对话结束时，没有记录表明考虑过哪些证据，无法从出错的地方重新运行分析，也无法比较两次运行，除非阅读两份对话转录。本 notebook 用 agent 写入的带类型的记录取代对话转录，加入检查，拒绝让证据薄弱的运行给出自信答案，并展示一旦该记录存在，它能带来什么可能。

**学习目标**：
- 将研究运行所依赖的状态写入带类型的记录，并与模型可见的内容分离
- 当证据太少、太旧，或日期晚于问题的截止时间时拒绝回答，并记录是哪项检查拒绝了
- 在运行中途将运行保存为 JSON 并恢复，使运行可以继续而不是重新开始
- 在移除一类证据的情况下重新运行已保存状态，以查明该证据是否发挥了作用

**书中参考**：第 24 章，24.3 节（Agent Memory: State, Persistence, and Replay）

**前置条件**：[`01_react_reasoning`](01_react_reasoning.ipynb)（providers）、[`02_tool_contracts`](02_tool_contracts.ipynb)（tools）。

In [1]:
"""Agent State, Memory, and Quality Gates: explicit, inspectable run state."""

import json
from datetime import date, datetime, timedelta

from agent_fixtures import get_demo_question
from agent_schemas import (
    AgentState,
    check_consistency_gate,
    check_coverage_gate,
    check_freshness_gate,
    run_quality_gates,
)

## 设置

`AS_OF_ISO` 是本次运行假定发生的时刻。所有依赖时间的量都以它为基准测量，而不是以系统时钟为基准，这正是本 notebook 能在今天和明年产生相同输出的原因。它被设为演示问题 2025-02-20 截止日期前一天，因此这次运行所处的时点，正是一个回答该问题的预测者会处的时点。

`RUN_ID` 为运行命名。checkpoint 只有在能与下一次运行区分开时才有用，这里使用固定值可保持打印输出稳定；真实部署会每次运行生成一个，未提供时 `AgentState` 就会这样做。

In [2]:
AS_OF_ISO = "2025-02-19T12:00:00"
RUN_ID = "ch24-state-demo"

In [3]:
as_of = datetime.fromisoformat(AS_OF_ISO)

## 为什么追踪记录（trace）不是状态（state）

[`01_react_reasoning`](01_react_reasoning.ipynb) 已经写入了一份持久记录：`RunTrace` 将问题、每次提示词和回复，以及每个搜索结果保存到 JSON 文件，而回放路径读取的正是该文件。因此运行是可复现的，读者也能准确看到发生了什么。

该记录无法做到的是参与运行。它在结束时从外部写入，保存的是模型的对话，而不是 agent 自己的结论。研究流程需要的四样东西仍然都缺失：

- **存放派生知识的地方。** agent 判断为相关的证据、尚未解决的问题、到目前为止做出的决定。这些都不是提示词或回复。
- **在回答之前可以检查的东西。** 门禁必须读取结构化的证据记录，而不是对话转录，并且必须在 agent 仍能对其采取行动时运行。
- **恢复点。** 对话日志可以回放一次运行；它不能让人从中途继续并更改设置。
- **可比较的对象。** 两次运行就是两段对话，而对散文做 diff 并不是一种度量。两条 `AgentState` 记录可以逐字段 diff。

因此 trace 和 state 是承担不同任务的不同产物，本章两者都保留：`RunTrace` 是运行从外部看起来的样子，`AgentState` 是 agent 从内部知道的内容。

## AgentState 数据结构

`agent_schemas.py` 中的 `AgentState` 包含八个字段，分为四组：

- **身份**：`run_id`、`question`、`cutoff_date` —— 这是哪次运行、它在回答什么，以及其证据不得晚于的发布日期
- **证据**：`evidence`，收集到的文档，以及 `open_questions`，agent 尚未解决的问题
- **溯源**：`tool_trace`，本次运行发起的每次工具调用，包括没有返回任何内容的调用
- **检查**：`quality_gates`，下文定义的检查及其结果，以及 `synthesis_status`，它从 `pending` 经 `in_progress` 变为 `complete`，或在门禁拒绝该运行时变为 `abstained`

它是一个普通 dataclass，带有 `to_json` 和 `from_json`，这正是一个 checkpoint 所需的全部。

In [4]:
question = get_demo_question()
state = AgentState(
    question=question.question,
    cutoff_date=question.cutoff_date,
    run_id=RUN_ID,
)
print(f"Run ID:         {state.run_id}")
print(f"Question:       {state.question}")
print(f"Cutoff:         {state.cutoff_date}")
print(f"Status:         {state.synthesis_status}")
print(f"Evidence items: {len(state.evidence)}")
print(f"Quality gates:  {len(state.quality_gates)}")

Run ID:         ch24-state-demo
Question:       Will NVIDIA beat Q4 FY2025 earnings expectations?
Cutoff:         2025-02-20
Status:         pending
Evidence items: 0
Quality gates:  0


## 收集证据

agent 运行的每次搜索都会追加两条记录：一条证据项，保存返回的内容；一条工具追踪条目，保存这次调用确实发生过。两者都保留很重要，因为一次没有返回任何内容的搜索不会改变证据列表，却会让 trace 多一条记录；只有 trace 才能区分“agent 没有查找”和“agent 查找了但什么也没找到”。

一条证据项包含 `type`、产生它的 `source`、检索时的 `timestamp`、产生它的 `query`，以及 `content` 本身。下面三项代表一次简短的研究会话，并逐条写出，以便各自的结构单独可见；在真实运行中，它们来自 [`02_tool_contracts`](02_tool_contracts.ipynb) 的搜索客户端。

第一项是对问题本身的搜索：现在关于被预测事件都有哪些说法。

In [5]:
search_evidence: list[dict] = []

search_evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA Q4 FY2025 earnings expectations",
        "content": {
            "results": [
                {
                    "title": "NVIDIA suppliers signal continued AI server demand",
                    "url": "https://wsj.com/nvidia-supply-chain",
                    "snippet": "Supply-chain checks point to resilient GPU demand ahead of earnings report.",
                    "published": "2025-02-17",
                },
                {
                    "title": "Analysts raise NVIDIA targets ahead of earnings",
                    "url": "https://nasdaq.com/nvidia-targets",
                    "snippet": "Street revisions remain constructive ahead of the Feb 26 report.",
                    "published": "2025-02-18",
                },
            ]
        },
    }
)

第二项是针对将决定结果的那条业务线的更窄查询。它携带相同的 `search_results` 类型：类型说明的是这属于何种证据，而不是由哪个查询产生。

In [6]:
search_evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA data center revenue growth",
        "content": {
            "results": [
                {
                    "title": "Data center revenue expected to exceed $30B",
                    "url": "https://reuters.com/nvidia-data-center",
                    "snippet": "Consensus estimates point to another record quarter for data center.",
                    "published": "2025-02-15",
                },
            ]
        },
    }
)

第三项是**基础比率（base rate）**：该事件在历史上发生的频率，与任何本季度的具体情况无关。它被单独赋予一个类型，因为下面的覆盖门禁要求它，原因值得直说。若仅凭当前报道进行预测，模型会依据眼前的叙事推理而忽略频率；而概率必须是从某个历史锚点（historical anchor）出发调整得到的。

In [7]:
search_evidence.append(
    {
        "type": "base_rate",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA historical earnings beat rate",
        "content": {
            "results": [
                {
                    "title": "NVIDIA has beaten earnings estimates 8 of last 8 quarters",
                    "url": "https://nasdaq.com/nvidia-earnings-history",
                    "snippet": "Strong track record of exceeding consensus, with average surprise of 12%.",
                    "published": "2025-02-10",
                },
            ]
        },
    }
)

将它们追加到状态中，就把每个证据项与其 trace 条目配对起来。

In [8]:
for item in search_evidence:
    state.evidence.append(item)
    state.tool_trace.append({"tool": "search", "query": item["query"], "status": "success"})

print(f"Evidence collected: {len(state.evidence)} items")
print(f"Search calls made: {len(state.tool_trace)}")
for item in state.evidence:
    n_results = len(item["content"].get("results", []))
    print(f'  [{item["type"]}] "{item["query"][:45]}" -> {n_results} results')

Evidence collected: 3 items
Search calls made: 3
  [search_results] "NVIDIA Q4 FY2025 earnings expectations" -> 2 results
  [search_results] "NVIDIA data center revenue growth" -> 1 results
  [base_rate] "NVIDIA historical earnings beat rate" -> 1 results


## 质量门禁

**质量门禁（quality gate）**是在允许 agent 产出答案之前对状态运行的检查。它是一份关于证据的契约：用代码陈述一次运行必须具备什么，其输出才值得一读。当某一门禁失败时，agent 会继续收集更多证据或弃权，并且失败会被记录下来，因此一次证据单薄的运行会明显表现为单薄，而非悄悄显得自信。

三个门禁覆盖证据不足的三种不同方式：

1. **覆盖度（Coverage）**询问是否存在足够多、且属于所需类型的证据。
2. **新鲜度（Freshness）**询问 agent 上一次去查找是在多久之前。
3. **一致性（Consistency）**询问它读到的任何内容是否有在截止时点之后发布的。

### 覆盖度门禁

两个条件。必需类型使该门禁成为关于一次预测需要*何种*证据的论证：`search_results` 用于当前正在发生的事，`base_rate` 用于该事物过去发生的频率。只获得当前报道的模型会跟随叙事而忽略频率，这正是典型的基础比率忽视（base-rate neglect），而要求两种类型是对此的一种廉价结构性防御。`min_items` 是另一个关于数量的问题：三条记录是本章的工作下限，低到一次正常运行能够通过，又高到单次搜索无法满足。

### 新鲜度门禁

该门禁读取每个证据项的 `timestamp`，这是 agent 检索文档的时间，而不是文档发布的时间。两者在两个方向上都会分离：一小时前抓取到一篇 2019 年论文的运行是新鲜的，而三天前抓取到今早报道的运行则不是。二十四小时是默认窗口，因为本章的问题会在新闻周期内解决；一个关于季度财报的问题则需要更宽的窗口。

未来的时间戳被视为失败，而不是最大程度的新鲜。它意味着时钟、fixture 或 checkpoint 有误，而一个把它读作新鲜的检查，恰恰会在记录不可信时让一次运行通过。

### 一致性门禁

新鲜度关乎运行；一致性关乎文档。每个搜索结果都必须带有可解析且早于问题截止时点的发布日期。结果失败有三种方式，它们会被分别报告，因为其含义不同：缺少日期是工具无法确定日期的证据，无法解析的日期是提供商 bug，而晚于截止时点的日期则是已经进入本次运行的事后信息（hindsight）。

[`02_tool_contracts`](02_tool_contracts.ipynb) 会在检索时进行过滤，因此在正常运行时不会有任何东西到达这里而失败。该门禁是对同一属性的第二道检查，位于流水线的另一端，它之所以存在，是因为第一道检查可能被绕过：证据会来自 fixture、来自 checkpoint，以及来自过滤器编写之后才出现的工具。

这三个函数与 `AgentState` 本身一起位于 `agent_schemas.py` 中，因此 [`04_research_agent`](04_research_agent.ipynb) 中的 agent 会依据此处示范的完全相同定义来检查其证据。同一个解析器处理截止时点和每个结果日期，因此截止时点比较会接受的日期不可能被结果比较拒绝；并且每个门禁会收集所有失败而非在第一个失败处停止，因此一次运行会报告问题的完整范围。

### 运行门禁

`run_quality_gates` 运行这三个门禁，将结果存储到状态上并返回它们，因此 checkpoint 不仅携带证据，还携带对证据作出的判断。失败时该做什么由调用方决定，而非由门禁决定：[`04_research_agent`](04_research_agent.ipynb) 会在一次已完成的 agent 运行上运行同样的这三个门禁，并报告它们对此运行的结论。

In [9]:
gates = run_quality_gates(state, as_of=as_of)

print("Quality gate results:")
for g in gates:
    status = "PASS" if g.passed else "FAIL"
    print(f"  [{status}] {g.gate_name}: {g.reason}")

all_passed = all(g.passed for g in gates)
print(f"\nAll gates passed: {all_passed}")

Quality gate results:
  [PASS] coverage: 3 items covering 2 types
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff

All gates passed: True


三项全部通过。本次运行在三条记录中收集到了两种必需证据类型，在声明的 as-of 时间检索到它们，并且它读到的每一份文档都在截止时点之前发布。通过门禁只说明证据可被采纳，仅此而已：它们中没有任何一个读过文档实际声称内容的哪怕一个词。

## 每个校验门能捕获什么

从未见其失败的校验门，是没人应该信任的校验门。下面的每个状态都被构造成恰好违反一个约定，因此失败可以归因于触发它的检查，而不是靠猜。

### 证据类型太少

一次搜索、一条结果、没有历史锚点。覆盖度校验门会先检查必需类型，再检查数量，因此这里被报告为缺少类型。

In [10]:
sparse_state = AgentState(question="one search only", cutoff_date="2025-02-20")
sparse_state.evidence.append(
    {
        "type": "search_results",
        "source": "web_search",
        "timestamp": as_of.isoformat(),
        "query": "NVIDIA Q4 FY2025 earnings expectations",
        "content": {"results": [{"title": "Single result", "published": "2025-02-15"}]},
    }
)

sparse_gates = run_quality_gates(sparse_state, as_of=as_of)
print("Gates on the one-search state:")
for g in sparse_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")

Gates on the one-search state:
  [FAIL] coverage: Missing evidence types: base_rate
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff


### 两种类型都有，但每种都太少

覆盖度是两个条件，而非一个。该状态满足类型要求却仍然失败，因为两条记录低于校验门配置所要求的最低数量。

In [11]:
thin_state = AgentState(question="both types, two records", cutoff_date="2025-02-20")
thin_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA Q4 FY2025 earnings expectations",
            "content": {"results": [{"title": "Supply chain check", "published": "2025-02-17"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA historical earnings beat rate",
            "content": {"results": [{"title": "Beat history", "published": "2025-02-10"}]},
        },
    ]
)

thin_gates = run_quality_gates(thin_state, as_of=as_of)
print("Gates on the two-record state:")
for g in thin_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")

Gates on the two-record state:
  [FAIL] coverage: Only 2 evidence items (need 3)
  [PASS] freshness: All evidence within 24h window
  [PASS] consistency: All result dates precede the cutoff


### 截止时间之后发布的文档

这是最重要、却最少出现的失败。该状态拥有两种类型的足够证据，检索也及时，但其中一条搜索结果发布于 2025-02-27：即 NVIDIA 发布财报后的第二天。读取它的智能体并不是在预测，而它的预测会看起来非常出色。

In [12]:
bad_state = AgentState(question="reads past the cutoff", cutoff_date="2025-02-20")
bad_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA Q4 FY2025 earnings expectations",
            "content": {
                "results": [
                    {"title": "Pre-cutoff article", "published": "2025-02-18"},
                    {"title": "Earnings beat confirmed", "published": "2025-02-27"},
                ]
            },
        },
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA data center revenue growth",
            "content": {"results": [{"title": "Data center outlook", "published": "2025-02-15"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": as_of.isoformat(),
            "query": "NVIDIA historical earnings beat rate",
            "content": {"results": [{"title": "Base rate data", "published": "2025-02-10"}]},
        },
    ]
)

bad_gates = run_quality_gates(bad_state, as_of=as_of)
print("Gates on the state that read past its cutoff:")
for g in bad_gates:
    print(f"  [{'PASS' if g.passed else 'FAIL'}] {g.gate_name}: {g.reason}")
    for issue in g.details.get("issues", []) if not g.passed else []:
        print(f"    -> {issue}")

Gates on the state that read past its cutoff:
  [PASS] coverage: 3 items covering 2 types
  [PASS] freshness: All evidence within 24h window
  [FAIL] consistency: 1 cutoff violations found
    -> Post-cutoff result: 'Earnings beat confirmed' (published 2025-02-27, cutoff 2025-02-20)


### 过期证据

新鲜度与一致性问的是不同的问题。一致性问的是文档是否在截止时间之前发布；新鲜度问的是智能体多久以前去查找过。一次两天前收集证据、现在才生成预测的运行，无论发布日期如何，读到的都是世界的过期快照。

下面的状态包含两种必需类型，且每份文档的日期都远早于截止时间，因此覆盖度和一致性通过。它的检索时间戳比声明的 as-of 时间早 48 小时，超过了校验门默认的 24 小时。

In [13]:
stale_ts = (as_of - timedelta(hours=48)).isoformat()
stale_state = AgentState(question="gathered two days ago", cutoff_date="2025-02-20")
stale_state.evidence.extend(
    [
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA earnings (stale)",
            "content": {"results": [{"title": "Old article", "published": "2025-02-12"}]},
        },
        {
            "type": "base_rate",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA beat rate (stale)",
            "content": {"results": [{"title": "Historical beats", "published": "2025-02-10"}]},
        },
        {
            "type": "search_results",
            "source": "web_search",
            "timestamp": stale_ts,
            "query": "NVIDIA guidance (stale)",
            "content": {"results": [{"title": "Guidance recap", "published": "2025-02-11"}]},
        },
    ]
)
stale_gates = run_quality_gates(stale_state, as_of=as_of)
print("Stale state gates:")
for g in stale_gates:
    status = "PASS" if g.passed else "FAIL"
    print(f"  [{status}] {g.gate_name}: {g.reason}")

Stale state gates:
  [PASS] coverage: 3 items covering 2 types
  [FAIL] freshness: 3 evidence items exceed 24h age limit
  [PASS] consistency: All result dates precede the cutoff


## 检查点

`AgentState` 是由普通列表和字符串组成的 dataclass，因此序列化它就是 `json.dumps(asdict(self))`，恢复它就是调用构造函数。这种简单性正是重点：一个需要自定义编码器的检查点格式，会在有人新增字段的那天悄悄丢掉该字段。

记录使用 JSON 而不是 pickle，原因与它存在于上下文窗口之外相同。pickle 至少需要此版本的这个代码库才能读取；JSON 检查点则可以在文本编辑器中打开、与另一次运行做 diff、无需加载 Python 即可查询，并在五年后由一个从未听说过 `AgentState` 的东西读取。

In [14]:
checkpoint = state.to_json()
print(f"Checkpoint size: {len(checkpoint):,} bytes")

data = json.loads(checkpoint)
print(f"Top-level fields: {list(data.keys())}")
print(f"Evidence items:   {len(data['evidence'])}")
print(f"Quality gates:    {len(data['quality_gates'])}")

Checkpoint size: 3,166 bytes
Top-level fields: ['question', 'cutoff_date', 'run_id', 'evidence', 'open_questions', 'tool_trace', 'quality_gates', 'synthesis_status']
Evidence items:   3
Quality gates:    3


恢复它必须返回同一条记录，而不只是相似的记录。比较两种序列化形式会一次性检查每个字段，包括手写比较会忘记查看的字段。

In [15]:
restored = AgentState.from_json(checkpoint)
print(f"Restored run ID:   {restored.run_id}")
print(f"Restored evidence: {len(restored.evidence)} items")
print(f"Restored gates:    {len(restored.quality_gates)} results")

assert restored.to_json() == checkpoint, "checkpoint did not survive the round trip"
print("Round trip: byte-identical")

Restored run ID:   ch24-state-demo
Restored evidence: 3 items
Restored gates:    3 results
Round trip: byte-identical


## 重放：没有历史锚点会发生什么

做检查点的原因不只是为了恢复运行。保存的状态也是实验的输入：恢复它、改变一件事、重新运行校验门，然后观察哪些结果发生了变化。下面的问题是，基础比率证据是承重的还是装饰性的。

In [16]:
print("=== Original run ===")
for g in state.quality_gates:
    print(f"  [{('PASS' if g.passed else 'FAIL')}] {g.gate_name}")

ablation = AgentState.from_json(checkpoint)
ablation.evidence = [e for e in ablation.evidence if e["type"] != "base_rate"]
ablation_gates = run_quality_gates(ablation, as_of=as_of)

print("\n=== Without base-rate evidence ===")
print(f"Evidence items: {len(ablation.evidence)} (was {len(state.evidence)})")
for g in ablation_gates:
    print(f"  [{('PASS' if g.passed else 'FAIL')}] {g.gate_name}")

=== Original run ===
  [PASS] coverage
  [PASS] freshness
  [PASS] consistency

=== Without base-rate evidence ===
Evidence items: 2 (was 3)
  [FAIL] coverage
  [PASS] freshness
  [PASS] consistency


In [17]:
if not all(g.passed for g in ablation_gates):
    ablation.synthesis_status = "abstained"
    print(f"Synthesis status: {ablation.synthesis_status}")
    print("No forecast is produced: the coverage contract is not satisfied.")
else:
    print("All gates still pass; the base rate was not required for coverage.")

Synthesis status: abstained
No forecast is produced: the coverage contract is not satisfied.


丢弃基础比率会使覆盖度校验门达不到所要求的类型，运行会弃权。值得争论的是这种行为，而不是代码：是否应该因为未找到历史频率就拒绝给出预测，这是一个研究决策；把它写成校验门，正是让它成为一个人们可以明确指出的决策，而不是智能体碰巧搜索了什么所导致的偶然结果。

## 这在记忆层级中的位置

通常会区分三种记忆，而它们很容易被混为一谈，因为语言模型会把三者都作为一段提示词中的文本呈现出来。**工作记忆**是上下文窗口：模型本轮能看到的内容，在 [`01_react_reasoning`](01_react_reasoning.ipynb) 中构建。**短期记忆**是运行自身持久化的记录，也就是上面的 `AgentState`：它比对话存在得更久，并且可以被模型之外的东西读取。**长期记忆**是跨运行携带的知识，按需检索，而不是保存在提示词中，这正是第 22 章用检索增强生成所构建的内容。

## 关键要点

1. **智能体可被审计所依据的状态必须位于上下文窗口之外。** 消息历史是一份对话转录，而非一条记录：它无法被查询、做 diff 或重放，并且对话结束后就会消失。
2. **门控是一份关于证据的契约，在综合之前检查。** 它用代码规定一次运行必须收集到什么才被允许给出答案，因此证据单薄的运行会弃权，而不是自信地猜测。
3. **将证据的检索时间与发布时间分开。** 二者回答不同的问题——本次运行是否过期，以及本次运行是否读取了未来信息——将二者混为一谈会让其中一个掩盖另一个。
4. **弃权是一种结果，而非失败。** 证据不支持的预测比没有预测代价更高，因为它会被当作一项判断来评分。
5. **检查点将消融实验变成单行实验。** 恢复检查点、丢弃一类证据、重新运行门控，并查看发生了什么变化。

**此处所构建内容的已知局限。** 门控检查证据是否存在、是否较新，以及是否早于截止时间；它们都不检查证据质量如何、五个结果究竟是五个来源还是同一篇通讯社报道被复制了五次，也不检查来源之间是否相互矛盾。新鲜度是相对于声明的 as-of 而非墙上时钟时间来衡量的，这使重放成为可能，也意味着一次过期的运行在重放时会显得新鲜。而且门控在综合之前只运行一次，因此在综合期间到达的证据不受门控约束。

**下一步**：[`04_research_agent`](04_research_agent.ipynb) 将提供方、搜索工具和此状态组合成一个智能体，该智能体收集证据、运行这些门控，然后要么预测，要么弃权。

**本书**：第 24.3 节深入介绍记忆层级，包括向量存储和基于 RAG 的长期记忆。